In [1]:
from pathlib import Path
import sys

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "medical_potential" / "config.py").is_file()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from medical_potential.config import BQ_DATASET_ID, MOA_INNOVATION_TABLE, PROJECT_ID
from medical_potential.gcp_utils import get_bq_client

bq_client = get_bq_client()
moa_innovation_table_id = f"{PROJECT_ID}.{BQ_DATASET_ID}.{MOA_INNOVATION_TABLE}"

moa_innovation_query = f"SELECT * FROM `{moa_innovation_table_id}`"
moa_innovation_df = bq_client.query(moa_innovation_query).to_dataframe()

moa_innovation_df.head()

c:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\.medical-potential-venv\Lib\site-packages\google\auth\transport\grpc.py:43: FutureWarning: grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, google-auth will raise its minimum requirements to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(


,drug_name,indication,mechanism_statement,moa_classification,score,guardrail,confidence_score,confidence_tier,justification_mechanism_summary,justification_novelty_vs_soc,...,justification_biological_rationale,justification_prior_validation_failure,justification_why_not_higher_score,narrative_rationale,sources_primary,sources_secondary,sources_tertiary,analysis_date,processing_time_seconds,created_at
0,Bofanglutide,"Type 2 Diabetes, Overweight, Obesity, Obstruct...",Glucagon Like Peptide 1 Receptor Agonist,Me-too,3,PASS,0.95,Tier 1,Bofanglutide (GZR18) is a long-acting peptide ...,This mechanism is not novel to the listed indi...,...,The biological rationale is exceptionally stro...,The GLP-1 mechanism has been successfully vali...,A higher score of 4 (Best-in-Class) was not gi...,Bofanglutide achieves a MoA Innovation score o...,"[""Signal Transduction and Targeted Therapy (20...","[""FDA Orange Book & Novel Drug Approvals (2024...","[""ClinicalTrials.gov (NCT06256562, NCT07622810...",2026-07-27,163.0,2026-07-27 15:34:33.053705+00:00
1,Bofanglutide,"Type 2 Diabetes, Overweight, Obesity, Obstruct...",Glucagon Like Peptide 1 Receptor Agonist,Me-too,3,PASS,0.95,Tier 1,Bofanglutide (GZR18) is a long-acting Glucagon...,The GLP-1 RA mechanism is not novel to any of ...,...,The biological rationale is exceptionally stro...,The GLP-1 mechanism is one of the most success...,A higher score of 4 (Best-in-Class) was not as...,Bofanglutide achieves a MoA Innovation score o...,"[""Signal Transduction and Targeted Therapy (20...","[""ADA Standards of Care in Diabetes 2026"", ""Cl...","[""FDA Orange Book"", ""AdisInsight Drug Profile:...",2026-07-28,100.4,2026-07-28 04:54:42.989602+00:00
2,Bofanglutide,"Type 2 Diabetes, Overweight, Obesity, Obstruct...",Glucagon Like Peptide 1 Receptor Agonist,Me-too,3,PASS,0.95,Tier 1,Bofanglutide (GZR18) is a long-acting Glucagon...,The mechanism is not novel to any of the liste...,...,The biological rationale is exceptionally stro...,The GLP-1 mechanism is a clinical success stor...,A score of 4 (Best-in-Class) was not assigned ...,Bofanglutide achieves a MoA Innovation score o...,"[""Signal Transduction and Targeted Therapy (20...","[""ClinicalTrials.gov (NCT06256562, NCT07493057...","[""FDA Orange Book"", ""Patsnap Synapse Drug Inte...",2026-07-29,108.6,2026-07-29 12:25:04.362534+00:00
3,Bgm-0504,"Type 2 Diabetes, Obesity, Overweight",Gastric Inhibitory Polypeptide Receptor Agonis...,Me-too,3,PASS,0.92,Tier 1,Bgm-0504 is a dual agonist targeting the Gastr...,This mechanism is not novel to the indications...,...,The biological rationale is exceptionally stro...,The mechanism is clinically validated with no ...,A score of 4 (Best-in-Class) was not assigned ...,Bgm-0504 achieves a MoA Innovation score of 3 ...,"[""Yuan, J., et al. (2024). Molecular dynamics-...","[""BrightGene Bio-Medical Technology Press Rele...","[""FDA Orange Book (Tirzepatide approval record...",2026-08-13,114.9,2026-08-13 13:15:33.877246+00:00
4,Bofanglutide,"Obesity, Type 2 Diabetes, Obstructive Sleep Ap...",Glucagon-like peptide-1 receptor agonist (GLP-...,Me-too,3,PASS,0.95,Tier 1,Bofanglutide (GZR18) is a long-acting peptide-...,The mechanism is not novel to the target indic...,...,The biological rationale is exceptionally stro...,The GLP-1 mechanism is one of the most success...,A higher score of 4 (Best-in-Class) was not as...,Bofanglutide achieves a MoA Innovation score o...,"[""Annals of Internal Medicine (2026) - Phase 2...","[""ClinicalTrials.gov (NCT06256562, NCT06728124...","[""HKEXnews / Sciwind Biosciences Global GLP-1 ...",2026-08-13,160.6,2026-08-13 13:35:10.823942+00:00


## Two types of evals can be implemented here

1. Run the whole code module using a stronger more reasoning level model and check how the output looks like

2. Validate the claims in the justification columns as the final output is based on these justifications

Note: Traditional RAGAS type of framework would be difficult to implement as there is o grounding data or metadata links available to implement those

In [ ]:
from medical_potential.config import CLAUDE_SONNET_MODEL
from medical_potential.claude_utils import get_claude_call_stats, reset_claude_call_stats
from medical_potential.moa_innovation.evals.bq_push import build_bq_rows, push_eval_rows
from medical_potential.moa_innovation.evals.claim_verification import run_claim_verification, summarize_claims
from medical_potential.moa_innovation.evals.classification_check import (
    check_classification_score,
    summarize_classification_check,
)
from medical_potential.moa_innovation.evals.llm_judge import run_llm_judge
from medical_potential.moa_innovation.evals.scoring import build_eval_scores
from medical_potential.moa_innovation.evals.utils import eval_timestamp, latest_rows

SELECTED_MOLECULES = [
 'Bgm-0504'
]
EVAL_MODEL = CLAUDE_SONNET_MODEL  # Claude on Vertex AI; claims are checked with Claude web search
EVAL_RUN_ID = eval_timestamp()
PUSH_TO_BQ = False
CLAIM_VERIFICATION_WEIGHT = 0.5
FAITHFULNESS_WEIGHT = 0.5

print(f"Loaded {len(moa_innovation_df):,} source rows")
print(f"Evaluation run: {EVAL_RUN_ID}")


Loaded 98 source rows
Evaluation run: 2026-10-29T15:23:41.349964+00:00


In [3]:
classification_results = check_classification_score(moa_innovation_df)
classification_summary = summarize_classification_check(classification_results)

print(classification_summary)
classification_failures = classification_results[
    ~classification_results["classification_check_pass"]
]
classification_failures[[
    "source_key",
    "source_classification",
    "source_score",
    "expected_score",
    "classification_check_reason",
]]

{'total_rows': 98, 'passed_rows': 98, 'failed_rows': 0, 'pass_rate': 1.0, 'failures_by_class': {}, 'score_crosstab': {1: {'Best-in-Class': 0, 'First-in-Class': 0, 'Me-too': 0, 'Poor/Invalid': 5, 'Weak/Outdated': 0}, 2: {'Best-in-Class': 0, 'First-in-Class': 0, 'Me-too': 0, 'Poor/Invalid': 0, 'Weak/Outdated': 9}, 3: {'Best-in-Class': 0, 'First-in-Class': 0, 'Me-too': 46, 'Poor/Invalid': 0, 'Weak/Outdated': 0}, 4: {'Best-in-Class': 30, 'First-in-Class': 0, 'Me-too': 0, 'Poor/Invalid': 0, 'Weak/Outdated': 0}, 5: {'Best-in-Class': 0, 'First-in-Class': 8, 'Me-too': 0, 'Poor/Invalid': 0, 'Weak/Outdated': 0}}}


,source_key,source_classification,source_score,expected_score,classification_check_reason


In [4]:
selected_rows = latest_rows(moa_innovation_df, SELECTED_MOLECULES)
selected_rows[["drug_name", "created_at", "moa_classification", "score"]]

,drug_name,created_at,moa_classification,score
13,Mazdutide,2026-08-21 08:58:39.742637+00:00,First-in-Class,5
12,Survodutide,2026-08-21 08:58:55.961079+00:00,Best-in-Class,4
18,Retatrutide,2026-08-21 09:01:10.359266+00:00,Best-in-Class,4
21,Tirzepatide,2026-08-21 09:01:11.852945+00:00,First-in-Class,5
23,Vk-2735,2026-08-21 09:03:31.339121+00:00,Me-too,3
24,Semaglutide,2026-08-21 09:04:03.141138+00:00,Me-too,3
75,Orforglipron Calcium,2026-09-21 05:31:01.875000+00:00,Me-too,3


In [5]:
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed


def evaluate_molecule(selected_row):
    molecule_name = str(selected_row["drug_name"])
    molecule_rows = selected_row.to_frame().T
    reset_claude_call_stats()
    evaluation_started_at = time.perf_counter()
    evaluation_run_status = "success"
    evaluation_error_message = None
    molecule_claim_rows = pd.DataFrame()
    molecule_claim_results = pd.DataFrame()
    molecule_judge_results = pd.DataFrame()

    try:
        molecule_claim_rows = run_claim_verification(molecule_rows, model=EVAL_MODEL)
        molecule_claim_results = summarize_claims(molecule_claim_rows)
        molecule_judge_results = run_llm_judge(molecule_rows, model=EVAL_MODEL)
        print(f"Completed evaluation for {molecule_name}")
    except Exception as error:
        evaluation_run_status = "failed"
        evaluation_error_message = str(error)
        print(f"Evaluation failed for {molecule_name}: {error}")
    finally:
        # Claude usage is not persisted to the Gemini stats table; print it instead.
        print(
            f"{molecule_name}: {evaluation_run_status} in "
            f"{time.perf_counter() - evaluation_started_at:.1f}s, usage={get_claude_call_stats()}"
        )

    return molecule_claim_rows, molecule_claim_results, molecule_judge_results


claim_row_frames = []
claim_result_frames = []
judge_result_frames = []

with ThreadPoolExecutor(max_workers=min(1, max(1, len(selected_rows)))) as executor:
    futures = [executor.submit(evaluate_molecule, selected_row) for _, selected_row in selected_rows.iterrows()]
    for future in as_completed(futures):
        molecule_claim_rows, molecule_claim_results, molecule_judge_results = future.result()
        if not molecule_claim_rows.empty:
            claim_row_frames.append(molecule_claim_rows)
        if not molecule_claim_results.empty:
            claim_result_frames.append(molecule_claim_results)
        if not molecule_judge_results.empty:
            judge_result_frames.append(molecule_judge_results)

claim_rows = (
    pd.concat(claim_row_frames, ignore_index=True)
    if claim_row_frames
    else pd.DataFrame()
)
claim_results = (
    pd.concat(claim_result_frames, ignore_index=True)
    if claim_result_frames
    else pd.DataFrame()
)
judge_results = (
    pd.concat(judge_result_frames, ignore_index=True)
    if judge_result_frames
    else pd.DataFrame()
)

print(f"Completed evaluations for {len(selected_rows)} molecule(s)")

[INFO] AFC is enabled with max remote calls: 10.
[WARNING] Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
[INFO] HTTP Request: POST https://aiplatform.googleapis.com/v1/projects/cognito-dev-380506/locations/global/publishers/google/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"



Gemini response served via mode: ON_DEMAND


[INFO] AFC is enabled with max remote calls: 10.
[INFO] HTTP Request: POST https://aiplatform.googleapis.com/v1/projects/cognito-dev-380506/locations/global/publishers/google/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"



Gemini response served via mode: ON_DEMAND


[INFO] AFC is enabled with max remote calls: 10.
[INFO] HTTP Request: POST https://aiplatform.googleapis.com/v1/projects/cognito-dev-380506/locations/global/publishers/google/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"



Gemini response served via mode: ON_DEMAND
Completed evaluation for Mazdutide


... (34 outputs left out)

[INFO] AFC is enabled with max remote calls: 10.
[INFO] HTTP Request: POST https://aiplatform.googleapis.com/v1/projects/cognito-dev-380506/locations/global/publishers/google/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"



Gemini response served via mode: ON_DEMAND
Completed evaluation for Orforglipron Calcium


[INFO] Appended 1 Gemini stats row(s) for Orforglipron Calcium/moa_innovation_evals to cognito-dev-380506.data_mart.gemini_model_api_call_logs


Completed evaluations for 7 molecule(s)


In [6]:
eval_results = build_eval_scores(
    classification_results,
    claim_results,
    judge_results,
    amber_threshold=65.0,
    green_threshold=85.0,
    claim_verification_weight=CLAIM_VERIFICATION_WEIGHT,
    faithfulness_weight=FAITHFULNESS_WEIGHT,
)

summary_columns = [
    "source_key",
    "source_classification",
    "source_score",
    "classification_check_pass",
    "claim_verification_score",
    "faithfulness_score_normalized",
    "composite_score",
    "eval_status",
]
eval_results[summary_columns]

,source_key,source_classification,source_score,classification_check_pass,claim_verification_score,faithfulness_score_normalized,composite_score,eval_status
0,Bofanglutide|2026-07-27 15:34:33.053705+00:00,Me-too,3,True,NaN,NaN,NaN,None
1,Bofanglutide|2026-07-28 04:54:42.989602+00:00,Me-too,3,True,NaN,NaN,NaN,None
2,Bofanglutide|2026-07-29 12:25:04.362534+00:00,Me-too,3,True,NaN,NaN,NaN,None
3,Bgm-0504|2026-08-13 13:15:33.877246+00:00,Me-too,3,True,NaN,NaN,NaN,None
4,Bofanglutide|2026-08-13 13:35:10.823942+00:00,Me-too,3,True,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...
93,Acmopatide|2026-09-21 07:10:36.099239+00:00,Weak/Outdated,2,True,NaN,NaN,NaN,None
94,Bgm-0504|2026-09-24 05:15:46.915785+00:00,Me-too,3,True,NaN,NaN,NaN,None
95,Vk-2375|2026-09-24 05:58:13.199817+00:00,Best-in-Class,4,True,NaN,NaN,NaN,None
96,Mdr-001|2026-09-28 12:57:04.947423+00:00,Best-in-Class,4,True,NaN,NaN,NaN,None


In [7]:
import pandas as pd

excel_output_path = repo_root / "medical_potential" / "moa_innovation" / "evals" / "output" / "moa_evaluation_results_final_wo_par_2.xlsx"
with pd.ExcelWriter(excel_output_path, engine="openpyxl") as writer:
    for dataframe, sheet_name in [
        (classification_results, "classification_results"),
        (claim_rows, "claim_results"),
        (judge_results, "judge_results"),
        (eval_results, "overall_results"),
    ]:
        export_dataframe = dataframe.copy()
        for column in export_dataframe.select_dtypes(include=["datetimetz"]).columns:
            export_dataframe[column] = export_dataframe[column].dt.tz_localize(None)
        export_dataframe.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Saved evaluation results to {excel_output_path}")

Saved evaluation results to c:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\medical_potential\moa_innovation\evals\output\moa_evaluation_results_final_wo_par_2.xlsx
